In [1]:
# LOAD
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder
import faiss
import numpy as np


chunks = np.load("Data/chunks.npy", allow_pickle=True)
index = faiss.read_index("Data/index.faiss")

embedding_model = SentenceTransformer("intfloat/multilingual-e5-small")
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

c:\Users\phili\Personal\University\Licenta\Proiect\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12258.10it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4376.27it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when l

In [2]:
# RETRIEVAL 
# retrieve top 20 chunks with the embedding model -> choose top 5 with the reranker model

def retrieve(query, k_initial=20, k_final=5):
    # Pasul A: Retrieval inițial (extrage mai multe fragmente)
    query_prefixed = "query: " + query
    q_emb = embedding_model.encode([query_prefixed])
    distances, indices = index.search(np.array(q_emb), k_initial)
    
    initial_chunks = [chunks[i] for i in indices[0]]
    
    # Pasul B: Reranking (Cross-Encoding)
    # Modelul compară perechi (întrebare, fragment) și dă un scor de relevanță
    sentence_combinations = [[query, chunk] for chunk in initial_chunks]
    rerank_scores = reranker_model.predict(sentence_combinations)
    
    # Pasul C: Sortăm fragmentele în funcție de scorul Reranker-ului
    reranked_results = sorted(
        list(zip(initial_chunks, rerank_scores)), 
        key=lambda x: x[1], 
        reverse=True
    )
    
    # Returnăm doar primele k_final cele mai bune
    return [res[0] for res in reranked_results[:k_final]]

QUESTION ANSWERING LLM

In [23]:
# QUESTION ANSWERING LLM
import requests

def ask_llm(question, context_chunks):
    context = "\n\n--- FRAGMENT NOU ---\n\n".join(context_chunks)

    prompt = f"""
Ești un asistent virtual oficial care ajută candidații cu informații despre admiterea la FMI UBB Cluj.
Trebuie să fii clar, concis și să răspunzi DOAR pe baza contextului oferit.

REGULI STRICTE:
1. Răspunde EXCLUSIV în limba română.
2. Folosește DOAR informațiile din secțiunea "Context". Nu folosi cunoștințele tale generale.
3. Dacă răspunsul nu se află în "Context", nu încerca să ghicești. Scrie EXACT următoarea propoziție: "Nu dețin această informație conform datelor extrase de pe site-ul oficial."
4. Nu inventa formule de calcul, date sau taxe.

Context:
{context}

Întrebare:
{question}

Răspuns:
"""

    response = requests.post(
        "http://localhost:1234/v1/chat/completions",
        json={
            "model": "google/gemma-4-e4b",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.1
        }
    )


    data = response.json()
    
    if "choices" not in data:
        print("Eroare primită de la LM Studio:", data)
        return "Eroare la generarea răspunsului."

    return data["choices"][0]["message"]["content"]

SCORING LLM

In [29]:
# SCORING SYSTEM LLM
def score_llm(question, context_chunks, answer):
    context = "\n\n--- FRAGMENT NOU ---\n\n".join(context_chunks)

    prompt = f"""
You are a ruthless and objective evaluator of a RAG (Retrieval-Augmented Generation) chatbot for university admissions.
Your task is to evaluate the chatbot's answer based STRICTLY on the provided Reference Context.

Question:
{question}

Chatbot Answer:
{answer}

Reference Context:
{context}

---
SCORING RUBRIC (Be extremely strict!):
1. Factual correctness (0-10): 
   - 10 = Perfectly matches the context.
   - 5 = Partially correct, but contains errors.
   - 0 = The information contradicts the context or is completely invented.
2. Completeness (0-10): Did it fully answer the user's prompt based on available context?
3. Clarity (0-10): Is the answer easy to read and in Romanian? (Score 0 if the answer is in English!).
4. Hallucinations penalty (0-10): 
   - 10 = NO hallucinations at all. Only used the context.
   - 0 = The model invented rules, formulas, or deadlines NOT present in the Reference Context.

Return ONLY valid JSON format:
{{
  "factual": <int>,
  "complete": <int>,
  "clarity": <int>,
  "no_hallucination": <int>,
  "final_score": <float between 0 and 10>,
  "brief_reason": "<1-2 sentences explaining exactly why points were deducted>"
}}
"""

    response = requests.post(
        "http://localhost:1234/v1/chat/completions",
        json={
            "model": "google/gemma-2-9b",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0
        }
    )


    data = response.json()
    
    if "choices" not in data:
        print("Eroare primită de la LM Studio:", data)
        return "Eroare la generarea răspunsului."

    return data["choices"][0]["message"]["content"]

In [5]:
# QA Model function
def answer_question(question, relevant_chunks):
    answer = ask_llm(question, relevant_chunks)
    return answer

# LLM-as-a-Judge Model function
def score_answer(question, relevant_chunks, answer):
    answer = score_llm(question, relevant_chunks, answer)
    return answer

Model answers question & Judge scores them

In [14]:
import time
import json
import pandas as pd


def load_questions():
    with open("../2nd_QAAndJudgeManualTests/Data/questions.json", "r", encoding="utf-8") as f:
        return json.load(f)


def save_results(experimentType, path, answers, scores):
    df = pd.DataFrame(answers)
    df.to_csv(experimentType + "/" + path + "/answers.csv", index=False, encoding="utf-8")

    df = pd.DataFrame(scores)
    df.to_csv(experimentType + "/" + path + "/scores.csv", index=False, encoding="utf-8")


def answerAndScore(path=None, initialK=20, finalK=5, question=None, experimentType="Manual Experiments"):
    questions = load_questions()
    answers = []
    scores = []
    totalTime = 0
    
    ct = 1
    if path != None and question == None:
        for q in questions:
            start = time.time()
            relevant_chunks = retrieve(q["question"], initialK, finalK)

            answer = answer_question(q["question"], relevant_chunks)
            print("Answered question ", ct) 
            answers.append({
                "id": q["id"],
                "question": q["question"],
                "category": q["category"],
                "difficulty": q["difficulty"],
                "answer": answer
            })

            score = score_answer(q, relevant_chunks, answer)
            print("Scored answer ", ct) ; ct+=1
            scores.append({
                "id": q["id"],
                "question": q["question"],
                "category": q["category"],
                "difficulty": q["difficulty"],
                "answer": answer,
                "score": score
            })

            elapsed = int(time.time() - start) ; minutes = elapsed // 60 ; seconds = elapsed % 60
            print(f"Time: {minutes}m{seconds}s")
            totalTime += elapsed

            save_results(experimentType, path, answers, scores)
            
        print(f"Total time: {totalTime//60}m{totalTime%60}s")


    elif path == None and question != None:
        relevant_chunks = retrieve(question, initialK, finalK)
        answer = answer_question(question, relevant_chunks)
        print(answer) 
        score = score_answer(question, relevant_chunks, answer)
        print("\n\n",score) ; ct+=1
    else:
        print("Wrong parameter input")

Analyse scores

In [35]:
import pandas as pd
import json


def parse_score(text):
    text = str(text).strip()

    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()

    return json.loads(text)


def analyse_scores(path, file="scores.csv", experimentType="Manual Experiments"):
    df = pd.read_csv(experimentType + "/" + path + "/" + file)

    df["score_dict"] = df["score"].apply(parse_score)

    df["final_score"] = df["score_dict"].apply(lambda x: x["final_score"])
    df["factual"] = df["score_dict"].apply(lambda x: x["factual"])
    df["complete"] = df["score_dict"].apply(lambda x: x["complete"])
    df["clarity"] = df["score_dict"].apply(lambda x: x["clarity"])

    print("Average final:", df["final_score"].mean())
    print("Average factual:", df["factual"].mean())
    print()
    print(df.groupby("difficulty")["final_score"].mean())
    print(df.groupby("category")["final_score"].mean())

Manual Experiments

In [21]:
#answerAndScore("1_newInput")
# Total time: 5m9s

#analyse_scores("1_newInput")

In [37]:
#answerAndScore("2_changedModels")
# Total time: 

# Improvement is very clear: the QA model answers much more reliably.

#analyse_scores("1_newInput")